# ClinVar / TCGA / 1000G tutorial

This notebook is a **tutorialized** version of the BRIDGE `variant_aware.py` entry point.

- The **code blocks below are copied verbatim** from `variant_aware.py` (i.e., code is consistent with the repository script).
- The notebook adds **step-by-step Markdown explanations**, plus **runnable demos** for parsing/mutation logic.
- End-to-end scoring requires a configured BRIDGE environment (dependencies, transformer path, and `.pth` checkpoints).

## CLI arguments

### Common arguments
| Argument | Type / choices | Required | Meaning |
|---|---:|:---:|---|
| `--variation_mode` | `before|after` | yes | Score reference window ('before') or ALT-substituted window ('after'). |
| `--fasta_sequence_path` | `PATH` | yes | Input FASTA containing window sequences. |
| `--variant_out_file` | `PATH` | yes | Output file (appended). |
| `--Transformer_path` | `PATH` | yes | Transformer path used by build_Transformer_embeddings. |
| `--model_save_path` | `PATH` | yes | Directory with BRIDGE checkpoints (*.pth). |
| `--device` | `cuda|cuda:N|cpu` | no | Torch device (default: cuda if available else cpu). |

### Pipeline selection flags
| Argument | Type / choices | Required | Meaning |
|---|---:|:---:|---|
| `--gwas` | `-` | no | Force GWAS pipeline (default if no pipeline flag provided). |
| `--ribosnitch/--ribosnitches` | `-` | no | Enable ribosnitch pipeline. |
| `--catalog_variants/--genomic_variants` | `-` | no | Enable catalog-variants pipeline (ClinVar/TCGA/1000G). |

### Catalog-variants-only arguments
| Argument | Type / choices | Required | Meaning |
|---|---:|:---:|---|
| `--model_id_strategy` | `from_header|from_fasta_stem` | no | Catalog mode: choose checkpoint naming strategy. |
| `--k` | `int` | no | Catalog mode: forwarded to build_Transformer_embeddings. |
| `--pos_weight` | `float` | no | Catalog mode: pos_weight for BCEWithLogitsLoss. |
| `--strict_ref_match` | `-` | no | Catalog mode: skip if REF/ALT cannot be matched in the window. |
| `--disable_off_by_one` | `-` | no | Catalog mode: disable +/-1 fallback when locating SNV index. |

## Input FASTA requirements (Catalog variants)

- Header must include `chr:start-end(strand)` and `POS:REF>ALT`.
- Default model id parsing expects `... <PROTEIN> in <CELL>`; otherwise set `--model_id_strategy from_fasta_stem`.


## Output format (Catalog variants)

```
<header_without_>\tmodel_id=<...>\tmode=<before|after>\tPrediction_score:<float>
```


## Script code

### Imports

In [1]:
# -*- coding: utf-8 -*-
from __future__ import annotations

import argparse
import logging
import os
import re
import sys
from pathlib import Path
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple, Dict, Optional, Callable

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertModel

repo_root = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(repo_root))

from utils.BRIDGE import BRIDGE
from utils.gen_transformer_embedding import build_Transformer_embeddings
from utils.train_loop import validate_without_sigmoid
from utils.utils import RBPInferDataset
from utils.FeatureEncoding import dealwithdata2
from utils.variant import read_fasta, open_output, parse_variant_block, apply_complement, substitute_base, ModelHub, parse_variant_block_flexible

/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Catalog pipeline section

In [2]:

# ClinVar / TCGA / 1000 Genomes style FASTA batches ("catalog variants")
# ---------------------------------------------------------------------------
# These datasets typically store variant windows directly in FASTA, where the header
# contains (chrom:start-end(strand)) and a SNV token like POS:REF>ALT.
#
# This branch is activated by passing `--catalog_variants` (alias: --genomic_variants).
# The default model naming strategy is `<PROTEIN>_<CELL>` parsed from the header.
# ---------------------------------------------------------------------------

_REGION_RE = re.compile(r"^(chr[^:]+):(\d+)-(\d+)\(([+-])\)")
_VAR_RE = re.compile(r"^(\d+):([ACGTN])>([ACGTN])$")


@dataclass
class ParsedHeader:
    "Parsed representation of a variant-window FASTA header."
    header_raw: str
    chrom: str
    start: int
    end: int
    strand: str
    var_pos: int
    ref: str
    alt: str
    protein: Optional[str]
    cell_line: Optional[str]


def parse_protein_cell_line(fields: List[str]) -> Tuple[Optional[str], Optional[str]]:
    "Heuristically parse protein & cell line from header tokens."
    protein: Optional[str] = None
    cell: Optional[str] = None

    # Prefer "... <PROTEIN> in <CELL>"
    if "in" in fields:
        # use the last "in" to be robust if "in" appears elsewhere
        idx_in = len(fields) - 1 - list(reversed(fields)).index("in")
        if idx_in + 1 < len(fields):
            cell = fields[idx_in + 1]
        if idx_in - 1 >= 0:
            protein = fields[idx_in - 1]

    # Fallbacks (match the old "[-3],[-1]" convention)
    if cell is None and len(fields) >= 1:
        cell = fields[-1]
    if protein is None:
        if len(fields) >= 3 and fields[-2] == "in":
            protein = fields[-3]
        elif len(fields) >= 3:
            protein = fields[-3]
        elif len(fields) >= 2:
            protein = fields[-2]

    return protein, cell


def parse_header_catalog(header_raw: str) -> ParsedHeader:
    "Parse a catalog-variant header with flexible token positions."
    fields = header_raw.split()

    region_tok: Optional[str] = None
    var_tok: Optional[str] = None

    for tok in fields:
        if tok.startswith("chr") and ":" in tok and "(" in tok and ")" in tok:
            region_tok = tok
            break
    if region_tok is None:
        for tok in fields:
            if _REGION_RE.match(tok):
                region_tok = tok
                break
    if region_tok is None:
        raise ValueError(f"Cannot find region token like chr:start-end(strand) in header: {header_raw}")

    for tok in fields:
        if _VAR_RE.match(tok):
            var_tok = tok
            break
    if var_tok is None:
        raise ValueError(f"Cannot find SNV token like POS:REF>ALT in header: {header_raw}")

    m_r = _REGION_RE.match(region_tok)
    if m_r is None:
        raise ValueError(f"Region token doesn't match expected pattern: {region_tok}")
    chrom, start, end, strand = m_r.group(1), int(m_r.group(2)), int(m_r.group(3)), m_r.group(4)

    m_v = _VAR_RE.match(var_tok)
    assert m_v is not None
    var_pos, ref, alt = int(m_v.group(1)), m_v.group(2), m_v.group(3)

    protein, cell = parse_protein_cell_line(fields)

    return ParsedHeader(
        header_raw=header_raw,
        chrom=chrom,
        start=start,
        end=end,
        strand=strand,
        var_pos=var_pos,
        ref=ref,
        alt=alt,
        protein=protein,
        cell_line=cell,
    )


def find_variant_index(
    seq: str,
    seq_start: int,
    var_pos: int,
    ref: str,
    alt: str,
    try_off_by_one: bool = True,
) -> Tuple[Optional[int], str]:
    "Locate the 0-based variant index inside the window sequence."
    candidates = [var_pos - seq_start]
    if try_off_by_one:
        candidates.append(var_pos - seq_start - 1)

    for idx0 in candidates:
        if idx0 < 0 or idx0 >= len(seq):
            continue
        base = seq[idx0]
        if base == ref:
            return idx0, "ref"
        if base == alt:
            return idx0, "alt"

    return None, "none"


def process_sequences_catalog_variants(
    headers: List[str],
    seqs: List[str],
    args: argparse.Namespace,
    hub: ModelHub,
) -> None:
    "Variant-aware scoring for ClinVar/TCGA/1000G-style FASTA batches (SNVs)."
    out_fp = open_output(args.variant_out_file)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(args.pos_weight), device=hub.device))

    with out_fp.open("a") as fout:
        for header_raw, seq in zip(headers, seqs):
            try:
                ph = parse_header_catalog(header_raw)
            except Exception as e:
                logging.error("[catalog_variants] Header parse failed: %s | %s", header_raw, e)
                continue

            ref, alt = ph.ref, ph.alt
            if ph.strand == "-":
                ref, alt = apply_complement(ref), apply_complement(alt)

            idx0, state = find_variant_index(
                seq=seq,
                seq_start=ph.start,
                var_pos=ph.var_pos,
                ref=ref,
                alt=alt,
                try_off_by_one=(not bool(args.disable_off_by_one)),
            )
            if state == "none":
                logging.warning("[catalog_variants] Cannot match REF/ALT at site: %s", ph.header_raw)
                if bool(args.strict_ref_match):
                    continue

            # choose checkpoint name
            if args.model_id_strategy == "from_fasta_stem":
                model_id = Path(args.fasta_sequence_path).stem
            else:
                if not ph.protein or not ph.cell_line:
                    logging.warning("[catalog_variants] Cannot parse protein/cell line for model id: %s", ph.header_raw)
                    continue
                model_id = f"{ph.protein}_{ph.cell_line}"

            model = hub.load_bridge(Path(args.model_save_path), model_id)
            if model is None:
                continue

            # build modified_seq depending on variation_mode and what we see in input
            if args.variation_mode == "before":
                if state == "alt":
                    logging.info("[catalog_variants] Input already ALT at site; scoring as-is in BEFORE: %s", ph.header_raw)
                modified_seq = seq
            else:  # after
                if idx0 is None:
                    modified_seq = seq
                elif state == "ref":
                    modified_seq = substitute_base(seq, idx0, alt)
                else:
                    modified_seq = seq

            emb, _ = build_Transformer_embeddings(
                sequences=[modified_seq],
                transformer_path=str(args.Transformer_path),
                device=hub.device,
                k=int(args.k),
                transpose_to_ch_first=True,
            )
            N = int(emb.shape[0])

            attn = np.zeros((N, 101, 103))
            struct = np.zeros((N, 1, 101))
            motif = np.zeros((N, 1, 101))
            biochem = dealwithdata2(modified_seq).transpose([0, 2, 1])

            dataset = RBPInferDataset(
                embedding=emb,
                attn=attn,
                struct=struct,
                motif=motif,
                biochem=biochem,
            )
            loader = DataLoader(dataset, batch_size=1, shuffle=False)

            score = validate_without_sigmoid(model, hub.device, loader, criterion).item()

            # Output keeps genomic_variants.py style (adds model_id/mode)
            fout.write(
                f"{ph.header_raw}\tmodel_id={model_id}"
                f"\tmode={args.variation_mode}\tPrediction_score:{score:.6f}\n"
            )


### CLI + main (listed; entrypoint guard shown separately)

In [3]:
def build_argparser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description=(
            "Variant-aware scoring with BRIDGE. Supports three pipelines:\n"
            "  (1) GWAS windows (legacy variant_aware.py behavior; default)\n"
            "  (2) Ribosnitch scoring (BRIDGE; per-record checkpoint selection)\n"
            "  (3) Catalog variants (ClinVar/TCGA/1000G-style FASTA batches; SNVs)\n"
        )
    )

    # ------------------------------------------------------------------
    # Common arguments (shared across pipelines)
    # ------------------------------------------------------------------
    parser.add_argument(
        "--variation_mode",
        choices=["before", "after"],
        required=True,
        help="Score sequences before variation (reference) or after variation (mutated).",
    )
    parser.add_argument("--fasta_sequence_path", required=True, type=Path)
    parser.add_argument("--variant_out_file", required=True, type=Path)
    parser.add_argument("--Transformer_path", required=True, type=Path)
    parser.add_argument("--model_save_path", required=True, type=Path)
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")

    # ------------------------------------------------------------------
    # Pipeline selection flags
    #
    # Backward compatibility:
    # - If you pass none of these flags, we default to GWAS mode.
    # - `--ribosnitches` is accepted as an alias for `--ribosnitch`.
    # - `--genomic_variants` is accepted as an alias for `--catalog_variants`.
    # ------------------------------------------------------------------
    pipe = parser.add_mutually_exclusive_group(required=False)
    pipe.add_argument(
        "--gwas",
        action="store_true",
        help="Force GWAS window scoring (default if no pipeline flag is provided).",
    )
    pipe.add_argument(
        "--ribosnitch",
        "--ribosnitches",
        dest="ribosnitch",
        action="store_true",
        help="Run ribosnitch scoring (BRIDGE).",
    )
    pipe.add_argument(
        "--catalog_variants",
        "--genomic_variants",
        dest="catalog_variants",
        action="store_true",
        help="Run ClinVar/TCGA/1000G-style FASTA batch scoring (SNVs).",
    )

    # ------------------------------------------------------------------
    # Ribosnitch-specific options
    # ------------------------------------------------------------------
    parser.add_argument(
        "--ribosnitch_after_variation",
        "--ribosnitches_after_variation",
        dest="ribosnitch_after_variation",
        action="store_true",
        help="Force ALT substitution for ribosnitch scoring, regardless of --variation_mode.",
    )
    parser.add_argument(
        "--ribosnitch_out_dir",
        "--ribosnitches_out_dir",
        dest="ribosnitch_out_dir",
        type=Path,
        default=Path("./results/ribosnitches"),
        help="Root output directory for ribosnitch results.",
    )

    # ------------------------------------------------------------------
    # Catalog-variants options (also used by genomic_variants.py)
    # ------------------------------------------------------------------
    parser.add_argument(
        "--model_id_strategy",
        choices=["from_header", "from_fasta_stem"],
        default="from_header",
        help=(
            "How to choose checkpoint name for catalog variants. "
            "from_header: <PROTEIN>_<CELL> parsed from header; "
            "from_fasta_stem: use FASTA filename stem."
        ),
    )
    parser.add_argument(
        "--k",
        type=int,
        default=1,
        help="K-mer / stride parameter forwarded to build_Transformer_embeddings (catalog variants branch).",
    )
    parser.add_argument(
        "--pos_weight",
        type=float,
        default=2.0,
        help="Positive class weight for BCEWithLogitsLoss (catalog variants branch).",
    )
    parser.add_argument(
        "--strict_ref_match",
        action="store_true",
        help="If set, skip records when REF/ALT cannot be matched inside the window (catalog variants branch).",
    )
    parser.add_argument(
        "--disable_off_by_one",
        action="store_true",
        help="Disable +/-1 position fallback when locating the SNV inside the window (catalog variants branch).",
    )

    return parser

def main() -> None:
    args = build_argparser().parse_args()
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
    device = torch.device(args.device)

    logging.info("Loading FASTA from %s", args.fasta_sequence_path)
    headers, sequences = read_fasta(args.fasta_sequence_path)

    # Pipeline selection validation.
    # - Default: GWAS (for backward compatibility) when no pipeline flag is provided.
    selected = []
    if bool(getattr(args, "gwas", False)):
        selected.append("gwas")
    if bool(getattr(args, "ribosnitch", False)) or bool(getattr(args, "ribosnitch_after_variation", False)):
        selected.append("ribosnitch")
    if bool(getattr(args, "catalog_variants", False)):
        selected.append("catalog_variants")

    if len(selected) > 1:
        raise SystemExit(f"Conflicting pipeline flags: {selected}. Please choose only one.")

    pipeline = selected[0] if selected else "gwas"

    hub = ModelHub(args.Transformer_path, device)

    if pipeline == "catalog_variants":
        logging.info("Running catalog variants pipeline (%s_variation)", args.variation_mode)
        process_sequences_catalog_variants(headers, sequences, args, hub)

    logging.info("Finished. Results appended to %s", args.variant_out_file)

In [ ]:
def process_sequences_catalog_variants(
    headers: List[str],
    seqs: List[str],
    args: argparse.Namespace,
    hub: ModelHub,
) -> None:
    "Variant-aware scoring for ClinVar/TCGA/1000G-style FASTA batches (SNVs)."
    out_fp = open_output(args.variant_out_file)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(args.pos_weight), device=hub.device))

    with out_fp.open("a") as fout:
        for header_raw, seq in zip(headers, seqs):
            try:
                ph = parse_header_catalog(header_raw)
            except Exception as e:
                logging.error("[catalog_variants] Header parse failed: %s | %s", header_raw, e)
                continue

            ref, alt = ph.ref, ph.alt
            if ph.strand == "-":
                ref, alt = apply_complement(ref), apply_complement(alt)

            idx0, state = find_variant_index(
                seq=seq,
                seq_start=ph.start,
                var_pos=ph.var_pos,
                ref=ref,
                alt=alt,
                try_off_by_one=(not bool(args.disable_off_by_one)),
            )
            if state == "none":
                logging.warning("[catalog_variants] Cannot match REF/ALT at site: %s", ph.header_raw)
                if bool(args.strict_ref_match):
                    continue

            # choose checkpoint name
            if args.model_id_strategy == "from_fasta_stem":
                model_id = Path(args.fasta_sequence_path).stem
            else:
                if not ph.protein or not ph.cell_line:
                    logging.warning("[catalog_variants] Cannot parse protein/cell line for model id: %s", ph.header_raw)
                    continue
                model_id = f"{ph.protein}_{ph.cell_line}"

            model = hub.load_bridge(Path(args.model_save_path), model_id)
            if model is None:
                continue

            # build modified_seq depending on variation_mode and what we see in input
            if args.variation_mode == "before":
                if state == "alt":
                    logging.info("[catalog_variants] Input already ALT at site; scoring as-is in BEFORE: %s", ph.header_raw)
                modified_seq = seq
            else:  # after
                if idx0 is None:
                    modified_seq = seq
                elif state == "ref":
                    modified_seq = substitute_base(seq, idx0, alt)
                else:
                    modified_seq = seq

            emb, _ = build_Transformer_embeddings(
                sequences=[modified_seq],
                transformer_path=str(args.Transformer_path),
                device=hub.device,
                k=int(args.k),
                transpose_to_ch_first=True,
            )
            N = int(emb.shape[0])

            attn = np.zeros((N, 101, 103))
            struct = np.zeros((N, 1, 101))
            motif = np.zeros((N, 1, 101))
            biochem = dealwithdata2(modified_seq).transpose([0, 2, 1])

            dataset = RBPInferDataset(
                embedding=emb,
                attn=attn,
                struct=struct,
                motif=motif,
                biochem=biochem,
            )
            loader = DataLoader(dataset, batch_size=1, shuffle=False)

            score = validate_without_sigmoid(model, hub.device, loader, criterion).item()

            # Output keeps genomic_variants.py style (adds model_id/mode)
            fout.write(
                f"{ph.header_raw}\tmodel_id={model_id}"
                f"\tmode={args.variation_mode}\tPrediction_score:{score:.6f}\n"
            )


In [ ]:
from pathlib import Path
import os
import logging
import torch

def find_bridge_root(start: Path) -> Path:
    # Search upwards for typical BRIDGE repo markers
    for p in [start, *start.parents]:
        if (p / "variant_aware.py").exists() and (p / "utils" / "variant.py").exists():
            return p
    raise FileNotFoundError(
        f"Cannot locate BRIDGE repo root from: {start}\n"
        f"Expected to find: variant_aware.py and utils/variant.py in the repo root."
    )

# Locate repo root robustly (works even if notebook cwd is not the repo root)
BRIDGE_HOME = find_bridge_root(Path.cwd())
os.chdir(BRIDGE_HOME)

# Keep paths consistent with your bash script
TRANSFORMER_PATH = str((BRIDGE_HOME / "RBPformer").resolve())
MODEL_SAVE_PATH = str((BRIDGE_HOME / "results" / "model").resolve())
# Ensure output directory exists
out_dir = BRIDGE_HOME / "results" / "variant" / "mut_before_after_score"
out_dir.mkdir(parents=True, exist_ok=True)

dataset = "1000genomes"
fasta = str((BRIDGE_HOME / "dataset_variant" / "1000genomes_diff.all.fa").resolve())

for mode in ["before", "after"]:
    out = str((out_dir / f"{dataset}.{mode}.txt").resolve())

    args = build_argparser().parse_args([
        "--catalog_variants",
        "--variation_mode", mode,
        "--fasta_sequence_path", fasta,
        "--variant_out_file", out,
        "--Transformer_path", TRANSFORMER_PATH,
        "--model_save_path", MODEL_SAVE_PATH,
        "--device", "cuda:0",  # switch to "cpu" if needed
    ])

    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
    device = torch.device(args.device)

    headers, sequences = read_fasta(args.fasta_sequence_path)
    hub = ModelHub(args.Transformer_path, device)
    process_sequences_catalog_variants(headers, sequences, args, hub)


Some weights of BertModel were not initialized from the model checkpoint at /fs1/private/user/wangyubo/code/BRIDGE/RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertModel were not initialized from the model checkpoint at /fs1/private/user/wangyubo/code/BRIDGE/RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertModel were not initialized from the model checkpoint at /fs1/private/user/wangyubo/code/BRIDGE/RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertModel were not initialized

: 